# 10+ Reasons To Like DuckDB

This notebook only contains SQL commands executed from an in-memory DuckDB database.

## GitHub API

Endpoint: <https://docs.github.com/en/rest/releases/releases?apiVersion=2026-03-10>.

Parsing the API response (sample with [DuckDB](https://api.github.com/repos/duckdb/duckdb/releases)):

### From The Command Line

Using [JQ](https://jqlang.org/).

In [1]:
%%bash

curl --silent 'https://api.github.com/repos/duckdb/duckdb/releases' \
| jq --raw-output '.[]
    | [(
        (.html_url | split("/") | .[4]),
        .tag_name,
        .name,
        .prerelease,
        .published_at
    )]
    | @csv'

"duckdb","v2.0.0","DuckDB v2.0.0 ""Cyanoptera""",false,"2026-10-21T12:49:12Z"
"duckdb","v1.5.6","DuckDB v1.5.6 Bugfix Release",false,"2026-09-28T13:35:11Z"
"duckdb","v1.5.5","DuckDB v1.5.5 Bugfix Release",false,"2026-07-22T10:51:50Z"
"duckdb","v1.5.4","DuckDB v1.5.4 Bugfix Release",false,"2026-06-17T10:45:53Z"
"duckdb","v1.4.5","DuckDB v1.4.5 Bugfix Release",false,"2026-06-17T10:42:38Z"
"duckdb","v1.5.3","DuckDB v1.5.3 Bugfix Release",false,"2026-05-20T11:50:43Z"
"duckdb","v1.5.2","DuckDB v1.5.2 Bugfix Release",false,"2026-04-13T10:46:51Z"
"duckdb","v1.5.1","DuckDB v1.5.1 Bugfix Release",false,"2026-03-23T12:10:58Z"
"duckdb","v1.5.0","DuckDB v1.5.0 ""Variegata""",false,"2026-03-09T12:40:59Z"
"duckdb","v1.4.4","v1.4.4 Bugfix Release",false,"2026-01-26T12:08:09Z"
"duckdb","v1.4.3","v1.4.3 Bugfix Release",false,"2025-12-09T10:20:40Z"
"duckdb","v1.4.2","v1.4.2 Bugfix Release",false,"2025-11-12T13:04:29Z"
"duckdb","v1.4.1","v1.4.1 Bugfix Release",false,"2025-10-07T10:29:57Z"
"duckdb","v1.4.

### In Python

Using [purejq](https://pypi.org/project/purejq/).

In [2]:
import purejq
import requests

with requests.get('https://api.github.com/repos/duckdb/duckdb/releases') as response:
    csv = purejq.all_outputs(
        """.[]
        | [(
            (.html_url | split("/") | .[4]),
            .tag_name,
            .name,
            .prerelease,
            .published_at
        )]
        | @csv""",
        response.json(),
    )

for row in csv:
    print(row)

"duckdb","v2.0.0","DuckDB v2.0.0 ""Cyanoptera""",false,"2026-10-21T12:49:12Z"
"duckdb","v1.5.6","DuckDB v1.5.6 Bugfix Release",false,"2026-09-28T13:35:11Z"
"duckdb","v1.5.5","DuckDB v1.5.5 Bugfix Release",false,"2026-07-22T10:51:50Z"
"duckdb","v1.5.4","DuckDB v1.5.4 Bugfix Release",false,"2026-06-17T10:45:53Z"
"duckdb","v1.4.5","DuckDB v1.4.5 Bugfix Release",false,"2026-06-17T10:42:38Z"
"duckdb","v1.5.3","DuckDB v1.5.3 Bugfix Release",false,"2026-05-20T11:50:43Z"
"duckdb","v1.5.2","DuckDB v1.5.2 Bugfix Release",false,"2026-04-13T10:46:51Z"
"duckdb","v1.5.1","DuckDB v1.5.1 Bugfix Release",false,"2026-03-23T12:10:58Z"
"duckdb","v1.5.0","DuckDB v1.5.0 ""Variegata""",false,"2026-03-09T12:40:59Z"
"duckdb","v1.4.4","v1.4.4 Bugfix Release",false,"2026-01-26T12:08:09Z"
"duckdb","v1.4.3","v1.4.3 Bugfix Release",false,"2025-12-09T10:20:40Z"
"duckdb","v1.4.2","v1.4.2 Bugfix Release",false,"2025-11-12T13:04:29Z"
"duckdb","v1.4.1","v1.4.1 Bugfix Release",false,"2025-10-07T10:29:57Z"
"duckdb","v1.4.

## DuckDB Friendly SQL

In [3]:
%config SqlMagic.displaylimit = 100
%config SqlMagic.autopandas = False

In [4]:
%%capture
%load_ext sql

In [5]:
# https://duckdb.org/docs/lts/guides/python/jupyter
%sql duckdb:///:memory: --alias duckdb

Connecting to 'duckdb'

In [6]:
%%sql

-- (1) Variables for clarity and flexibility
SET VARIABLE repositories=[
  -- (2) Support of lists and dictionaries
  -- ⚠️ Use single-quotes to enclose strings, and double-quotes to enclose table fields
  {'account': 'duckdb',    'repo': 'duckdb'},
  {'account': 'microsoft', 'repo': 'vscode'},
  {'account': 'redis',     'repo': 'redis'},
  {'account': 'elastic',   'repo': 'elasticsearch'},
  {'account': 'pydantic',  'repo': 'pydantic'},
];

Running query in 'duckdb'

Success


In [7]:
%%sql

-- (3) Macro (scalar output)
CREATE OR REPLACE MACRO get_url(r) AS
-- (8) Support of HTTP protocol (but also S3, GCS, etc.)
-- The extension httpfs is used behind the scene
-- Responses are cached for optimized re-run (HEAD + GET)
-- GitHub REST API endpoints for releases:
-- https://docs.github.com/en/rest/releases/releases?apiVersion=2026-03-10
'https://api.github.com/repos/' || r['account'] || '/' || r['repo'] || '/releases';
-- Sample URL: https://api.github.com/repos/duckdb/duckdb/releases

Running query in 'duckdb'

Count


In [8]:
%%sql

-- (4) Macro (table output)
CREATE OR REPLACE MACRO releases(repositories) AS TABLE
SELECT
  -- (9) JSON extraction
  -- (10) Handy text functions
  string_split(json->>'$.html_url', '/')[5] AS repository,
  json->>'$.tag_name' AS tag_name,
  json->>'$.name' AS "name",
  json->>'$.prerelease' AS prerelease,
  -- (11) Friendly SQL: trailing comma allowed - More on https://duckdb.org/docs/current/sql/dialect/friendly_sql
  DATE (json->>'$.published_at') AS published_at,
-- (5) Support of JSON format
FROM read_json_objects(
  -- (6) Multiple data sources support, with list or glob pattern
  -- (7) List comprehensions
  [get_url(r) for r in repositories]
  -- (7') Lambdas
  -- apply(getvariable('repositories'), lambda r: get_url(r))
  -- (7'') Lambdas
  -- apply(getvariable('repositories'), get_url)
)
WHERE prerelease = false;

Running query in 'duckdb'

Count


In [9]:
%%sql

CREATE OR REPLACE TABLE frequency AS
-- (12) Common Table Expressions
WITH
  stats AS (
    SELECT
      repository,
      published_at,
      -- (13) Window functions (release - previous release, for every repository)
      (published_at - lag(published_at) OVER (
        PARTITION BY repository ORDER BY published_at)
      ) AS gap,
    FROM releases($repositories)
  )
SELECT
  repository,
  -- (14) Friendly SQL: casting
  avg(gap)::INTEGER AS days,
FROM stats
-- (15) Friendly SQL: no need to repeat all grouping columns
GROUP BY ALL;

Running query in 'duckdb'

Count


In [10]:
%%sql

-- (16) Friendly SQL: no useless "SELECT *"
FROM frequency
ORDER BY days DESC;

Running query in 'duckdb'

repository,days
duckdb,37
pydantic,19
elasticsearch,6
redis,4
vscode,4


In [11]:
%%sql

SELECT
  repository,
  -- (17) Some convenient functions for fancy output
  bar(days, 0, 100) as bar,
FROM frequency
ORDER BY days ASC
-- (18) Friendly SQL: limit can be expressed in percentage
LIMIT 80%;

Running query in 'duckdb'

repository,bar
redis,███▏
vscode,███▏
elasticsearch,████▊
pydantic,███████████████▏


`-- (19) Last but not least: it can be executed in WASM` (check the [Web Shell](https://shell.duckdb.org/)).